# A10. Brand Consistency Audit Notebook

> **Related Module**: [A10 Brand Building](../paths/a-operators/a10-brand-building.md)
>
> **Features**: Input content from various platforms → AI compares consistency → Scoring + Improvement suggestions
>
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kangise/ecommerce-ai-roadmap/blob/main/notebooks/a10-brand-audit.ipynb)

In [ ]:
!pip install -q openai pandas

## 1. Input Brand Content from Each Platform

In [ ]:
import os
from openai import OpenAI
import pandas as pd
import json

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', 'your-key-here')
client = OpenAI(api_key=OPENAI_API_KEY)

# === Replace with your real content ===
BRAND_CONTENT = {
    'brand_name': 'SoundPeak',
    'brand_voice': 'Professional yet approachable, tech-savvy, quality-focused',
    'platforms': {
        'amazon_listing': {
            'title': 'SoundPeak Pro Wireless Earbuds - Active Noise Cancellation, 36H Battery, IPX5 Waterproof',
            'bullet_1': 'PREMIUM SOUND QUALITY: 13mm dynamic drivers deliver deep bass and crystal-clear highs.',
            'description': 'Experience premium audio with SoundPeak Pro wireless earbuds.'
        },
        'shopify_product': {
            'title': 'SoundPeak Pro — Your Daily Audio Companion',
            'description': 'Designed for the modern listener. SoundPeak Pro combines cutting-edge noise cancellation with all-day comfort.',
        },
        'instagram_bio': 'Premium audio for everyday life 🎧 | ANC + 36H Battery | Free shipping',
        'tiktok_bio': 'Sound that moves with you 🔥 Earbuds that actually work',
        'email_welcome': 'Welcome to the SoundPeak family! You just made the best audio decision of your life.'
    }
}

print(f'Brand: {BRAND_CONTENT["brand_name"]}')
print(f'Platforms to audit: {len(BRAND_CONTENT["platforms"])}')
for p in BRAND_CONTENT['platforms']:
    print(f'  ✓ {p}')

## 2. AI Brand Consistency Audit

In [ ]:
prompt = f"""You are a brand consistency auditor. Analyze the following brand content across platforms.

Brand: {BRAND_CONTENT['brand_name']}
Intended Brand Voice: {BRAND_CONTENT['brand_voice']}

Content by platform:
{json.dumps(BRAND_CONTENT['platforms'], indent=2)}

Please audit and return JSON:
{{
  "overall_score": 1-10,
  "voice_consistency": {{
    "score": 1-10,
    "issues": ["list of inconsistencies"],
    "recommendations": ["list of fixes"]
  }},
  "messaging_consistency": {{
    "score": 1-10,
    "issues": ["list of inconsistencies"],
    "recommendations": ["list of fixes"]
  }},
  "platform_specific": {{
    "amazon": {{"score": 1-10, "feedback": "..."}},
    "shopify": {{"score": 1-10, "feedback": "..."}},
    "instagram": {{"score": 1-10, "feedback": "..."}},
    "tiktok": {{"score": 1-10, "feedback": "..."}},
    "email": {{"score": 1-10, "feedback": "..."}}
  }},
  "top_3_actions": ["prioritized action items"]
}}"""

response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': prompt}],
    response_format={'type': 'json_object'},
    max_tokens=1500
)

audit = json.loads(response.choices[0].message.content)
print(f'\n=== Brand Consistency Audit: {BRAND_CONTENT["brand_name"]} ===')
print(f'Overall Score: {audit["overall_score"]}/10')
print(f'Voice Consistency: {audit["voice_consistency"]["score"]}/10')
print(f'Messaging Consistency: {audit["messaging_consistency"]["score"]}/10')

print(f'\n--- Platform Scores ---')
for platform, data in audit.get('platform_specific', {}).items():
    emoji = '✅' if data['score'] >= 7 else ('⚠️' if data['score'] >= 5 else '❌')
    print(f'{emoji} {platform}: {data["score"]}/10 — {data["feedback"]}')

print(f'\n--- Top 3 Actions ---')
for i, action in enumerate(audit.get('top_3_actions', []), 1):
    print(f'{i}. {action}')

## 3. Export

In [ ]:
with open('brand_audit_report.json', 'w') as f:
    json.dump(audit, f, indent=2)
print('Exported to brand_audit_report.json')

# Simple CSV export
rows = [{'Platform': k, 'Score': v['score'], 'Feedback': v['feedback']} for k, v in audit.get('platform_specific', {}).items()]
pd.DataFrame(rows).to_csv('brand_audit_scores.csv', index=False)
print('Exported to brand_audit_scores.csv')